# Validation check visualization

This notebook runs the opt-in `delaunay` binary to generate deterministic validation-level failure examples. It then renders the binary artifact into a 3-column table: generated picture, public check/test reference, and explanation.

Generated files are written under `target/notebooks/01_validation_check_visualization/`.

## 1. Generate validation cases

The CLI produces the case geometry and diagnostics. The notebook only renders that artifact.

In [ ]:
"""Generate deterministic validation failure cases with the local delaunay binary."""

import json
import math
import os
import shutil
import subprocess
import textwrap
import time
from pathlib import Path
from typing import Any, cast

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon


def find_repo_root(start: Path) -> Path:
    """Return the repository root containing the project marker files."""
    for path in (start, *start.parents):
        if (path / "Cargo.toml").is_file() and (path / "pyproject.toml").is_file():
            return path
    message = "Run this notebook from inside the delaunay repository."
    raise RuntimeError(message)


def positive_int_env(name: str, default: int) -> int:
    """Read a positive integer environment override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be a positive integer, got {raw_value!r}") from error
    if value <= 0:
        raise ValueError(f"{name} must be positive, got {value}")
    return value


def run_command(command: list[str], *, cwd: Path, timeout: int) -> subprocess.CompletedProcess[str]:
    """Run one repository command with captured output and a timeout."""
    print("$", " ".join(command))
    result = subprocess.run(  # noqa: S603 - command is an argv list with shell=False and controlled cwd.
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        message = f"command failed with exit code {result.returncode}: {' '.join(command)}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        raise RuntimeError(message)
    return result


def delaunay_command_prefix(root: Path) -> list[str]:
    """Return the command prefix for the local `delaunay` CLI."""
    configured = os.environ.get("DELAUNAY_BINARY")
    if configured is not None:
        if not configured.strip():
            message = "DELAUNAY_BINARY must not be empty"
            raise ValueError(message)
        binary = Path(configured).expanduser()
        binary = binary if binary.is_absolute() else root / binary
        if not binary.is_file():
            raise FileNotFoundError(f"DELAUNAY_BINARY does not point to a file: {binary}")
        return [str(binary)]

    binary_name = "delaunay.exe" if os.name == "nt" else "delaunay"
    local_binary = root / "target" / "perf" / binary_name
    if local_binary.is_file():
        return [str(local_binary)]

    cargo = shutil.which("cargo")
    if cargo is None:
        message = "cargo executable was not found on PATH; set DELAUNAY_BINARY to a built binary"
        raise RuntimeError(message)
    return [cargo, "run", "--profile", "perf", "--features", "cli", "--bin", "delaunay", "--"]


ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_STEM = "01_validation_check_visualization"
OUTPUT_DIR = ROOT / "target" / "notebooks" / NOTEBOOK_STEM
DEMO_PATH = OUTPUT_DIR / "validation_demo.json"
TABLE_FIGURE_PATH = OUTPUT_DIR / "validation_model_failures.png"
TIMEOUT_SECONDS = positive_int_env("DELAUNAY_VALIDATION_DEMO_TIMEOUT_SECONDS", 120)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    *delaunay_command_prefix(ROOT),
    "validation-demo",
    "--output",
    str(DEMO_PATH),
]

start_time = time.perf_counter()
run_command(command, cwd=ROOT, timeout=TIMEOUT_SECONDS)
elapsed = time.perf_counter() - start_time

print(f"Repository: {ROOT}")
print(f"Validation demo JSON: {DEMO_PATH}")
print(f"Generated validation demo cases in {elapsed:.2f}s")

## 2. Load and check the artifact

The parser validates the small JSON contract before plotting so the rendered table tracks the binary schema.

In [ ]:
"""Load and validate the generated validation-demo JSON artifact."""

type JsonObject = dict[str, Any]
type Point2 = tuple[float, float]


def reject_json_constant(value: str) -> object:
    """Reject non-standard JSON numeric constants before artifact parsing."""
    raise ValueError(f"JSON artifact contains non-finite value {value}")


def load_json_object(path: Path) -> JsonObject:
    """Load a JSON object from disk."""
    with path.open(encoding="utf-8") as handle:
        data = json.load(handle, parse_constant=reject_json_constant)
    if not isinstance(data, dict):
        raise TypeError(f"expected JSON object in {path}")
    return cast("JsonObject", data)


def require_object(value: Any, context: str) -> JsonObject:
    """Return a JSON object or raise a contextual type error."""
    if not isinstance(value, dict):
        raise TypeError(f"{context} must be a JSON object")
    return cast("JsonObject", value)


def require_list(value: Any, context: str) -> list[Any]:
    """Return a JSON list or raise a contextual type error."""
    if not isinstance(value, list):
        raise TypeError(f"{context} must be a list")
    return value


def require_str(value: Any, context: str) -> str:
    """Return a JSON string or raise a contextual type error."""
    if not isinstance(value, str):
        raise TypeError(f"{context} must be a string")
    return value


def require_number(value: Any, context: str) -> float:
    """Return a finite JSON number as float or raise a contextual error."""
    if isinstance(value, bool) or not isinstance(value, int | float):
        raise TypeError(f"{context} must be a finite JSON number")
    number = float(value)
    if not math.isfinite(number):
        raise ValueError(f"{context} must be finite, got {number!r}")
    return number


def require_int(value: Any, context: str) -> int:
    """Return a JSON integer or raise a contextual type error."""
    if type(value) is not int:
        raise TypeError(f"{context} must be an integer")
    return value


def point_coordinates(record: JsonObject, context: str) -> Point2:
    """Parse one 2D point coordinate array."""
    coordinates = require_list(record.get("coordinates"), f"{context}.coordinates")
    if len(coordinates) != 2:
        raise ValueError(f"{context}.coordinates must have length 2")
    return (require_number(coordinates[0], f"{context}.x"), require_number(coordinates[1], f"{context}.y"))


def parse_case(raw_case: Any, index: int) -> JsonObject:
    """Validate the fields each notebook-rendered case needs."""
    case = require_object(raw_case, f"cases[{index}]")
    visual = require_object(case.get("visual"), f"cases[{index}].visual")
    require_int(case.get("level"), f"cases[{index}].level")
    for field in ("layer", "title", "status", "public_check", "public_reference", "input_summary", "explanation", "diagnostic"):
        require_str(case.get(field), f"cases[{index}].{field}")
    points = require_list(visual.get("points"), f"cases[{index}].visual.points")
    for point_index, raw_point in enumerate(points):
        point = require_object(raw_point, f"cases[{index}].visual.points[{point_index}]")
        require_str(point.get("label"), f"cases[{index}].visual.points[{point_index}].label")
        point_coordinates(point, f"cases[{index}].visual.points[{point_index}]")
    for field in ("simplices", "highlighted_simplices", "highlighted_edges", "invalid_points", "isolated_points", "duplicate_simplices"):
        require_list(visual.get(field), f"cases[{index}].visual.{field}")
    return case


validation_demo = load_json_object(DEMO_PATH)
if validation_demo.get("schema") != "delaunay.validation_demo":
    raise ValueError(f"unexpected schema: {validation_demo.get('schema')!r}")
if validation_demo.get("schema_version") != 1:
    raise ValueError(f"unexpected schema version: {validation_demo.get('schema_version')!r}")
if validation_demo.get("dimension") != 2:
    raise ValueError(f"expected 2D validation demo, got {validation_demo.get('dimension')!r}")

valid_baseline = parse_case(validation_demo.get("valid_baseline"), -1)
validation_cases = [parse_case(raw_case, index) for index, raw_case in enumerate(require_list(validation_demo.get("cases"), "cases"))]
levels = [require_int(case.get("level"), f"cases[{index}].level") for index, case in enumerate(validation_cases)]
if levels != [1, 2, 3, 4, 5]:
    raise ValueError(f"expected validation levels [1, 2, 3, 4, 5], got {levels}")

print(f"baseline: {valid_baseline['title']} -> {valid_baseline['diagnostic']}")
for case in validation_cases:
    print(f"Level {case['level']}: {case['layer']} -> {case['status']}")

## 3. Render the validation table

Each picture is generated from the CLI artifact. The middle column points at the public API check and test anchor; the right column gives the failure interpretation and exact binary diagnostic.

In [ ]:
"""Render generated validation cases as a three-column visual table."""


def case_visual(case: JsonObject) -> JsonObject:
    """Return a validated case visual object."""
    return require_object(case.get("visual"), "case.visual")


def visual_points(visual: JsonObject) -> list[tuple[str, Point2]]:
    """Return labeled 2D points from visual metadata."""
    parsed_points: list[tuple[str, Point2]] = []
    for index, raw_point in enumerate(require_list(visual.get("points"), "visual.points")):
        point = require_object(raw_point, f"visual.points[{index}]")
        parsed_points.append((require_str(point.get("label"), f"visual.points[{index}].label"), point_coordinates(point, f"visual.points[{index}]")))
    return parsed_points


def int_list(value: Any, context: str) -> list[int]:
    """Parse a JSON integer list."""
    return [require_int(item, f"{context}[{index}]") for index, item in enumerate(require_list(value, context))]


def simplex_list(value: Any, context: str) -> list[list[int]]:
    """Parse a JSON simplex list."""
    return [int_list(item, f"{context}[{index}]") for index, item in enumerate(require_list(value, context))]


def edge_list(value: Any, context: str) -> list[tuple[int, int]]:
    """Parse a JSON edge list."""
    edges: list[tuple[int, int]] = []
    for index, raw_edge in enumerate(require_list(value, context)):
        edge = int_list(raw_edge, f"{context}[{index}]")
        if len(edge) != 2:
            raise ValueError(f"{context}[{index}] must contain exactly two point indices")
        edges.append((edge[0], edge[1]))
    return edges


def circle_value(value: Any) -> tuple[Point2, float] | None:
    """Parse an optional finite positive circumcircle witness."""
    if value is None:
        return None
    circle = require_object(value, "visual.circumcircle")
    center_raw = require_list(circle.get("center"), "visual.circumcircle.center")
    if len(center_raw) != 2:
        message = "visual.circumcircle.center must have length 2"
        raise ValueError(message)
    center = (require_number(center_raw[0], "visual.circumcircle.center.x"), require_number(center_raw[1], "visual.circumcircle.center.y"))
    radius = require_number(circle.get("radius"), "visual.circumcircle.radius")
    if radius <= 0.0:
        raise ValueError(f"visual.circumcircle.radius must be positive, got {radius}")
    return (center, radius)


def point_by_index(points: list[Point2], index: int, context: str) -> Point2:
    """Return a point by visual index with contextual bounds checking."""
    if index < 0 or index >= len(points):
        raise IndexError(f"{context} index {index} is outside 0..{len(points) - 1}")
    return points[index]


def simplex_points(points: list[Point2], simplex: list[int], context: str) -> list[Point2]:
    """Return the three points for a 2D simplex visual."""
    if len(simplex) != 3:
        raise ValueError(f"{context} must contain exactly three point indices")
    return [point_by_index(points, point_index, f"{context}[{offset}]") for offset, point_index in enumerate(simplex)]


type VisualIndexData = tuple[set[int], list[tuple[int, int]], set[int], set[int], list[list[int]]]


def require_simplex_index(index: int, simplex_count: int, context: str) -> int:
    """Return a simplex index with contextual bounds checking."""
    if index < 0 or index >= simplex_count:
        raise IndexError(f"{context} index {index} is outside 0..{simplex_count - 1}")
    return index


def visual_index_data(visual: JsonObject, points: list[Point2], simplices: list[list[int]]) -> VisualIndexData:
    """Parse and validate visual index lists against point and simplex arrays."""
    highlighted_simplex_indices = int_list(visual.get("highlighted_simplices"), "visual.highlighted_simplices")
    highlighted_edges = edge_list(visual.get("highlighted_edges"), "visual.highlighted_edges")
    invalid_point_indices = int_list(visual.get("invalid_points"), "visual.invalid_points")
    isolated_point_indices = int_list(visual.get("isolated_points"), "visual.isolated_points")
    duplicate_simplices = simplex_list(visual.get("duplicate_simplices"), "visual.duplicate_simplices")

    highlighted_simplices = {
        require_simplex_index(index, len(simplices), f"visual.highlighted_simplices[{offset}]") for offset, index in enumerate(highlighted_simplex_indices)
    }
    for offset, point_index in enumerate(invalid_point_indices):
        point_by_index(points, point_index, f"visual.invalid_points[{offset}]")
    for offset, point_index in enumerate(isolated_point_indices):
        point_by_index(points, point_index, f"visual.isolated_points[{offset}]")
    return (highlighted_simplices, highlighted_edges, set(invalid_point_indices), set(isolated_point_indices), duplicate_simplices)


def axes_limits(points: list[Point2], circle: tuple[Point2, float] | None) -> tuple[float, float, float, float]:
    """Return padded axis limits that include points and optional circle."""
    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    if circle is not None:
        center, radius = circle
        xs.extend([center[0] - radius, center[0] + radius])
        ys.extend([center[1] - radius, center[1] + radius])
    min_x = min(xs) if xs else -1.0
    max_x = max(xs) if xs else 1.0
    min_y = min(ys) if ys else -1.0
    max_y = max(ys) if ys else 1.0
    span = max(max_x - min_x, max_y - min_y, 1.0)
    padding = 0.18 * span
    center_x = (min_x + max_x) / 2.0
    center_y = (min_y + max_y) / 2.0
    half_span = span / 2.0 + padding
    return (center_x - half_span, center_x + half_span, center_y - half_span, center_y + half_span)


def draw_visual(ax: Any, case: JsonObject) -> None:
    """Draw one validation case from generated visual metadata."""
    visual = case_visual(case)
    labeled_points = visual_points(visual)
    points = [point for _, point in labeled_points]
    simplices = simplex_list(visual.get("simplices"), "visual.simplices")
    highlighted_simplices, highlighted_edges, invalid_points, isolated_points, duplicate_simplices = visual_index_data(
        visual,
        points,
        simplices,
    )
    circle = circle_value(visual.get("circumcircle"))

    palette = ["#7dd3fc", "#fca5a5", "#86efac", "#fde68a", "#c4b5fd"]
    for simplex_index, simplex in enumerate(simplices):
        polygon_points = simplex_points(points, simplex, f"visual.simplices[{simplex_index}]")
        facecolor = "#fb7185" if simplex_index in highlighted_simplices else palette[simplex_index % len(palette)]
        edgecolor = "#be123c" if simplex_index in highlighted_simplices else "#334155"
        polygon = Polygon(
            polygon_points,
            closed=True,
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=1.6,
            alpha=0.34 if simplex_index in highlighted_simplices else 0.24,
        )
        ax.add_patch(polygon)

    for duplicate_index, duplicate in enumerate(duplicate_simplices):
        duplicate_points = simplex_points(points, duplicate, f"visual.duplicate_simplices[{duplicate_index}]")
        polygon = Polygon(
            duplicate_points,
            closed=True,
            facecolor="none",
            edgecolor="#7f1d1d",
            linewidth=2.0,
            hatch="///",
            alpha=0.8,
        )
        ax.add_patch(polygon)
        centroid_x = sum(point[0] for point in duplicate_points) / len(duplicate_points)
        centroid_y = sum(point[1] for point in duplicate_points) / len(duplicate_points)
        ax.text(centroid_x, centroid_y, "x2", ha="center", va="center", fontsize=9, weight="bold", color="#7f1d1d")

    for edge_index, (left, right) in enumerate(highlighted_edges):
        left_point = point_by_index(points, left, f"visual.highlighted_edges[{edge_index}][0]")
        right_point = point_by_index(points, right, f"visual.highlighted_edges[{edge_index}][1]")
        ax.plot([left_point[0], right_point[0]], [left_point[1], right_point[1]], color="#111827", linewidth=2.4)

    if circle is not None:
        center, radius = circle
        ax.add_patch(Circle(center, radius, fill=False, linestyle="--", linewidth=1.6, edgecolor="#f97316", alpha=0.9))
        ax.scatter([center[0]], [center[1]], s=18, color="#f97316", zorder=5)

    for index, (label, point) in enumerate(labeled_points):
        marker = "x" if index in invalid_points else "o"
        color = "#dc2626" if index in invalid_points else "#0f172a"
        size = 58 if index in invalid_points else 42
        ax.scatter([point[0]], [point[1]], marker=marker, s=size, color=color, zorder=6)
        if index in isolated_points:
            ax.scatter([point[0]], [point[1]], marker="o", s=180, facecolor="none", edgecolor="#dc2626", linewidth=1.8, zorder=5)
        ax.text(point[0] + 0.03, point[1] + 0.03, label, fontsize=9, weight="bold", color=color)

    limits = axes_limits(points, circle)
    ax.set_xlim(limits[0], limits[1])
    ax.set_ylim(limits[2], limits[3])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_facecolor("#f8fafc")


def wrapped_text(value: str, width: int) -> str:
    """Wrap long table text for stable figure layout."""
    return textwrap.fill(value, width=width, break_long_words=True)


def shorten_text(value: str, limit: int) -> str:
    """Shorten a diagnostic while preserving its first concrete payload."""
    if len(value) <= limit:
        return value
    return f"{value[: limit - 1].rstrip()}..."


fig = plt.figure(figsize=(15.0, 12.0), facecolor="white", layout="constrained")
grid = fig.add_gridspec(len(validation_cases), 3, width_ratios=[1.15, 1.25, 2.25], wspace=0.08, hspace=0.22)
headers = ["Generated failure picture", "Public check / test", "Explanation"]

for row_index, case in enumerate(validation_cases):
    visual_axis = fig.add_subplot(grid[row_index, 0])
    check_axis = fig.add_subplot(grid[row_index, 1])
    explanation_axis = fig.add_subplot(grid[row_index, 2])

    if row_index == 0:
        for axis, header in zip((visual_axis, check_axis, explanation_axis), headers, strict=True):
            axis.set_title(header, fontsize=12, weight="bold", pad=12)

    draw_visual(visual_axis, case)
    visual_axis.text(
        0.03,
        0.97,
        f"Level {case['level']}: {case['layer']}",
        transform=visual_axis.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        weight="bold",
        color="#0f172a",
        bbox={"boxstyle": "round,pad=0.2", "facecolor": "#f8fafc", "edgecolor": "none", "alpha": 0.85},
    )

    for axis in (check_axis, explanation_axis):
        axis.set_axis_off()

    check_text = f"{case['public_check']}\n\n{case['public_reference']}"
    check_axis.text(0.0, 0.82, wrapped_text(check_text, 34), ha="left", va="top", fontsize=9.5, color="#0f172a")

    explanation = f"{case['title']}\n\n{case['explanation']}\n\nDiagnostic: {shorten_text(str(case['diagnostic']), 360)}"
    explanation_axis.text(0.0, 0.88, wrapped_text(explanation, 72), ha="left", va="top", fontsize=9.5, color="#111827")

TABLE_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(TABLE_FIGURE_PATH, dpi=180, facecolor=fig.get_facecolor())
plt.show()

print(f"validation table figure: {TABLE_FIGURE_PATH}")

## 4. Copyable artifact summary

Use these paths when linking generated outputs from documentation or issue comments.

In [ ]:
"""Print the generated validation-demo artifact paths."""

if not DEMO_PATH.is_file():
    raise FileNotFoundError(f"validation demo JSON was not written: {DEMO_PATH}")
if not TABLE_FIGURE_PATH.is_file():
    raise FileNotFoundError(f"validation table figure was not written: {TABLE_FIGURE_PATH}")

print(f"JSON artifact:   {DEMO_PATH}")
print(f"Table artifact:  {TABLE_FIGURE_PATH}")